# Notebook 03 — Advanced Cardiotoxicity: hERG + CiPA Framework
**Author: Himanshu Goel** | [Website](https://himanshugoel.github.io)

---

## Why Cardiac Safety Matters

Drug-induced cardiac arrhythmia is the **#1 cause of post-market drug withdrawal**.
The primary culprit is blockade of the **hERG potassium channel** (IKr), which
prolongs ventricular repolarisation → long QT syndrome → torsades de pointes → sudden death.

The **CiPA initiative** (FDA/HESI/CSRC, 2013–present) moves beyond single-channel
hERG screening to a three-channel integrated model that better predicts clinical risk:

| Channel | Gene | Role in cardiac AP | Risk direction |
|---------|------|--------------------|----------------|
| hERG (IKr) | KCNH2 | Repolarisation phase 3 | Blockade → prolonged QT |
| Nav1.5 (INa) | SCN5A | Depolarisation phase 0 | Blockade → partially **protective** |
| Cav1.2 (ICaL) | CACNA1C | Plateau phase 2 | Blockade → partially **protective** |

**Key insight:** Verapamil blocks hERG strongly but is clinically safe because
it equally blocks Cav1.2, shortening the plateau and cancelling the QT effect.

---

## Workflow Overview

```
Dataset (27 CiPA reference compounds)
  │
  ├─ Step 1: Featurise  ──► ECFP4 fingerprint (2048 bits)
  │                         + physicochemical descriptors (10 values)
  │
  ├─ Step 2: EDA        ──► IC50 distributions, class balance check
  │
  ├─ Step 3: Regression ──► log10(hERG IC50) — RF, SVR, XGBoost + 5-fold CV
  │
  ├─ Step 4: CiPA Rule  ──► Multi-channel blockade score (Hill equation)
  │
  ├─ Step 5: ML Class.  ──► 3-class risk (High/Medium/Low) — RF + 3-fold OOF
  │
  └─ Step 6: Uncertainty ──► Bootstrap ensemble epistemic uncertainty
```

**References:**
- Crumb et al. (2016) *J. Pharmacol. Toxicol. Methods* — CiPA reference dataset
- Fermini et al. (2016) *JAHA* — CiPA initiative overview
- Gintant et al. (2016) *Nat. Rev. Drug Discov.* — Evolution of cardiac safety testing


In [ ]:
# Install required packages (skip if already in your environment)
!pip install rdkit scikit-learn pandas numpy matplotlib xgboost -q


---
## Step 1 — Imports, Dataset & Featurisation

### Feature design
Each molecule is represented by two complementary feature types:

| Feature | Size | What it captures |
|---------|------|------------------|
| **ECFP4** (Morgan r=2) | 2 048 bits | Circular atom environments up to 4 bonds; encodes local topology and pharmacophores |
| **Physicochemical** | 10 values | MW, LogP, TPSA, Fsp³, MR, valence electrons, HBD, HBA, aromatic rings, total rings |

ECFP4 captures **structural similarity** (key for hERG blockers which share a
basic nitrogen + aromatic core scaffold). Physicochemical descriptors encode
**ADME-relevant properties** that correlate with membrane permeability and binding.

### Dataset: CiPA reference compounds
27 compounds from Crumb et al. (2016) with experimentally measured IC50 values
for all three cardiac channels. `IC50 = 100 µM` encodes "no relevant blockade"
(compound inactive at therapeutic concentrations).


In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

# ── Cheminformatics ───────────────────────────────────────────────────────────
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.DataStructs import ConvertToNumpyArray
RDLogger.DisableLog('rdApp.warning')   # suppress RDKit SMILES parse warnings

# ── Numerics & visualisation ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import (
    KFold, StratifiedKFold,
    cross_val_score, cross_validate,
)
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score, classification_report,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

# =============================================================================
# CiPA reference dataset — Crumb et al. 2016 + literature expansions
# =============================================================================
# Each row: (SMILES, name, hERG_IC50_uM, Nav1.5_IC50_uM, Cav1.2_IC50_uM, risk_class)
# IC50 = 100 uM means the channel is NOT relevantly blocked at therapeutic doses.
# Risk classes are the consensus CiPA labels: High / Medium / Low.

CIPA_DATA = [
    # ── HIGH RISK: potent hERG blockers with no protective counter-channel effect
    ('OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1',
     'Terfenadine',    0.090, 16.0,  7.40, 'High'),   # withdrawn antihistamine
    ('CCOC(=O)c1cc2cc(OC)c(OC)cc2[nH]1',
     'Cisapride',      0.012, 100,   100,  'High'),   # withdrawn GI drug
    ('CN(CCOc1ccc(NS(=O)(=O)c2ccc(NC)cc2)cc1)S(=O)(=O)c1ccc(N)cc1',
     'Dofetilide',     0.004, 100,   100,  'High'),   # antiarrhythmic, narrow TI
    ('OC(c1ccnc2ccccc12)C1CC2CCN1CC2C=C',
     'Quinidine',      0.294,  5.5,  32.0, 'High'),   # Na/K blocker but hERG dominates
    ('CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc21',
     'Chlorpromazine', 0.185,  3.8,   3.0, 'High'),   # antipsychotic, multi-channel
    ('OC1(c2ccc(Cl)cc2)CCN(CCCC(=O)c2ccc(F)cc2)CC1',
     'Haloperidol',    0.034, 100,   100,  'High'),   # antipsychotic
    ('O=C1Nc2ccccc2N1C1CCN(CCCC(c2ccc(F)cc2)c2ccc(F)cc2)CC1',
     'Pimozide',       0.018, 100,   100,  'High'),   # antipsychotic, QTc risk
    ('CN(C)CCCC1=NC2=CC=CC=C2C(=C1)C1=CC=CS1',
     'Thioridazine',   0.130,  3.5,  40.0, 'High'),   # antipsychotic, withdrawn
    ('COc1ccc(CCN2CCC(Nc3nc4ccc(F)cc4n3Cc3ccc(F)cc3)CC2)cc1',
     'Astemizole',     0.001, 100,   100,  'High'),   # most potent hERG blocker here
    ('Fc1ccc(C(=O)CCCN2CCC(O)(c3ccc(F)cc3)CC2)cc1',
     'Droperidol',     0.008, 100,   100,  'High'),   # antiemetic, black-box warning

    # ── MEDIUM RISK: mixed channel effects — context-dependent clinical outcome
    ('COc1ccc(CCN(C)CCC(C#N)(c2ccc(OC)c(OC)c2)C(C)C)cc1OC',
     'Verapamil',      0.143,  5.0,   0.22, 'Medium'), # hERG blocked but Cav1.2 offsets
    ('OCC(NC(=O)c1nc2cc(OCC(F)(F)F)ccc2c(OCC(F)(F)F)c1)C',
     'Flecainide',     1.090,  0.18,  40.0, 'Medium'), # Nav1.5 dominant blocker
    ('COc1ccc(OCC(O)CN2CC(=O)N(c3ccccc3F)CC2)cc1OC',
     'Ranolazine',    11.500,  7.5,   3.0,  'Medium'), # late INa blocker, anti-anginal
    ('CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1',
     'Tamoxifen',      0.560, 100,   100,   'Medium'), # breast cancer drug, moderate hERG
    ('CC1=CC(=O)c2ccccc2C1=O',
     'Menadione',      2.000, 100,   100,   'Medium'), # vitamin K analogue
    ('CCOC(=O)N1CCC(=C2c3ccccc3CC(n3ccnc3)c3ccccc32)CC1',
     'Loratadine',     5.000, 100,   100,   'Medium'), # non-sedating antihistamine

    # ── LOW RISK: weak hERG or strong counter-channel balance → clinically safe
    ('CC(O)CNc1ccc(NS(C)(=O)=O)cc1',
     'Sotalol',       127.0,  100,   100,   'Low'),    # beta-blocker, very weak hERG
    ('CC(Nc1c(C)cccc1C)COC',
     'Mexiletine',    100.0,    0.25, 100,   'Low'),    # Nav1.5 blocker, used as control
    ('COc1ccc(C2Sc3cc(OC)ccc3N(CCN(C)C)C(=O)C2OC(C)=O)cc1',
     'Diltiazem',       5.74, 100,     0.069, 'Low'),   # CCB: Cav1.2 offset dominant
    ('Cn1ccnc1CN1CCc2c(c3ccccc23)C1=O',
     'Ondansetron',     1.57, 100,   100,   'Low'),    # 5-HT3 antiemetic
    ('CCOC(=O)C1=C(C)NC(C)=C(C(=O)OCC)C1c1ccccc1[N+](=O)[O-]',
     'Nifedipine',     37.0,  100,     0.017, 'Low'),   # dihydropyridine CCB
    ('COCCc1ccc(OCC(O)CNC(C)C)cc1',
     'Metoprolol',    100.0,  100,   100,   'Low'),    # beta-blocker, hERG inactive
    ('CCN(CC)CC(=O)Nc1c(C)cccc1C',
     'Lidocaine',      50.0,    0.03, 100,   'Low'),    # Nav1.5 anaesthetic
    ('CCOC(=O)C1=C(COCCN)NC(C)=C(C(=O)OCC)C1c1ccc(Cl)cc1',
     'Amlodipine',     50.0,  100,     0.001, 'Low'),   # long-acting CCB
    ('CC(=O)Oc1ccccc1C(=O)O',
     'Aspirin',       100.0,  100,   100,   'Low'),    # negative control
    ('CC(C)Cc1ccc(C(C)C(=O)O)cc1',
     'Ibuprofen',     100.0,  100,   100,   'Low'),    # negative control (NSAID)
    ('Cn1cnc2c(=O)[nH]c(=O)n(C)c21',
     'Theophylline',  100.0,  100,   100,   'Low'),    # xanthine, negative control
]

# Physicochemical descriptor functions — accessed via getattr to avoid 10 separate lines
DESC_NAMES  = ('ExactMolWt', 'MolLogP', 'TPSA', 'FractionCSP3',
               'MolMR', 'NumValenceElectrons')
RDKIT_NAMES = ('CalcNumHBD', 'CalcNumHBA', 'CalcNumAromaticRings', 'CalcNumRings')

# All feature names in order — used later for feature importance labelling
FEAT_NAMES = (
    [f'ECFP4[{i}]' for i in range(2048)]
    + list(DESC_NAMES)
    + list(RDKIT_NAMES)
)  # total: 2058 features

# CiPA risk class → integer label mapping (required by sklearn classifiers)
RISK_MAP = {'High': 0, 'Medium': 1, 'Low': 2}
INV_MAP  = {v: k for k, v in RISK_MAP.items()}

# Consistent colours used in every figure throughout this notebook
RISK_COLORS = {'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#27ae60'}


def featurize(smiles: str, n_bits: int = 2048) -> np.ndarray | None:
    """
    Convert a SMILES string to a fixed-length feature vector.

    Returns
    -------
    np.ndarray of shape (2058,) = [ECFP4 bits | physicochemical descriptors]
    None if the SMILES cannot be parsed by RDKit.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # ECFP4 fingerprint: radius=2 means atom environments up to 4 bonds away
    # are hashed into n_bits buckets. Captures scaffold + key substituents.
    fp = np.zeros(n_bits, dtype=np.float32)
    ConvertToNumpyArray(
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits), fp
    )

    # Physicochemical descriptors: ADME-relevant properties
    physchem = np.array(
        [float(getattr(Descriptors,      fn)(mol)) for fn in DESC_NAMES] +
        [float(getattr(rdMolDescriptors, fn)(mol)) for fn in RDKIT_NAMES],
        dtype=np.float32
    )

    return np.concatenate([fp, physchem])  # shape: (2058,)


# ── Build dataset arrays ──────────────────────────────────────────────────────
raw_feats = [featurize(row[0]) for row in CIPA_DATA]
is_valid  = [f is not None for f in raw_feats]

# Keep only rows with parseable SMILES
cipa_v  = [row for row, ok in zip(CIPA_DATA, is_valid) if ok]
X       = np.stack([f for f, ok in zip(raw_feats, is_valid) if ok])  # (27, 2058)
names   = [row[1] for row in cipa_v]
y_risk  = [row[5] for row in cipa_v]

# log10 transform: linearises the 4-decade IC50 range (0.001 to 127 uM)
# so that regression MSE treats a 10x error the same at 0.01 uM and at 10 uM
y_herg  = np.log10([row[2] for row in cipa_v])
y_nav   = np.log10([row[3] for row in cipa_v])
y_cal   = np.log10([row[4] for row in cipa_v])
y_cls   = np.array([RISK_MAP[r] for r in y_risk])

# StandardScaler: zero-mean, unit-variance normalisation
# Required for SVR (which is sensitive to feature scale);
# also helps RF/XGBoost converge faster with mixed-scale features
scaler  = StandardScaler()
X_s     = scaler.fit_transform(X)

# ── Sanity checks ─────────────────────────────────────────────────────────────
n_failed = sum(not ok for ok in is_valid)
if n_failed:
    print(f'WARNING: {n_failed} SMILES failed RDKit parsing and were dropped.')

print(f'Dataset      : {len(cipa_v)} / {len(CIPA_DATA)} compounds')
print(f'Feature dim  : {X.shape[1]}  (ECFP4: 2048, physicochemical: 10)')
print(f'hERG IC50    : {10**y_herg.min():.4f} – {10**y_herg.max():.0f} µM  '
      f'({y_herg.max() - y_herg.min():.1f} log-decades)')
print(f'Risk classes : {dict(Counter(y_risk))}')


---
## Step 2 — Exploratory Data Analysis

Before building any model, we need to answer three diagnostic questions:

1. **Do IC50 values separate cleanly by risk class?**  
   If High-risk compounds have universally low IC50 values and Low-risk ones are high,
   simple thresholding might suffice. Mixed distributions justify ML.

2. **Is the dataset class-balanced?**  
   Class imbalance (e.g., 10 High vs 6 Medium) requires stratified cross-validation
   and balanced class weights to prevent the model from ignoring minority classes.

3. **Does the multi-channel picture look different from hERG alone?**  
   Nav1.5 and Cav1.2 IC50 distributions overlapping with hERG-only patterns would
   suggest the extra channels add no discriminative value.


In [ ]:
# =============================================================================
# Step 2 — Exploratory Data Analysis
# =============================================================================

CLASS_ORDER = ['High', 'Medium', 'Low']

# ── Figure 1: IC50 distributions per channel ─────────────────────────────────
# Box plot groups compounds by CiPA risk class and shows IC50 spread per channel.
# If the boxes don't overlap, the channel is a good discriminator.
# IC50=100 (log10=2) means 'not active' — many Low-risk compounds cluster here.

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

channel_specs = [
    ('hERG (IKr)',    y_herg, 'Primary arrhythmia driver'),
    ('Nav1.5 (INa)',  y_nav,  'Depolarisation — partially protective'),
    ('Cav1.2 (ICaL)', y_cal,  'Plateau phase — partially protective'),
]

for ax, (channel_name, y_arr, role) in zip(axes, channel_specs):
    # Group IC50 values by risk class for the boxplot
    groups = [
        [y for y, r in zip(y_arr, y_risk) if r == cls]
        for cls in CLASS_ORDER
    ]
    bp = ax.boxplot(
        groups,
        labels=CLASS_ORDER,
        patch_artist=True,   # fill boxes with colour
        notch=False,
        widths=0.5,
    )
    for patch, cls in zip(bp['boxes'], CLASS_ORDER):
        patch.set_facecolor(RISK_COLORS[cls])
        patch.set_alpha(0.75)

    # log10(1 uM) = 0 — the classic hERG safety threshold
    ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.5, label='IC50 = 1 µM')
    ax.set_title(f'{channel_name}\n{role}', fontsize=10)
    ax.set_ylabel('log₁₀(IC50 / µM)')
    ax.set_xlabel('CiPA Risk Class')
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(
    f'CiPA Dataset — IC50 Distributions  (n = {len(cipa_v)} compounds)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('eda_ic50_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Class balance check ─────────────────────────────────────────────
# Rule of thumb: if any class has fewer than 10% of total samples, use
# class_weight='balanced' in the classifier AND stratified CV splits.

counts = Counter(y_risk)
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(
    CLASS_ORDER,
    [counts[c] for c in CLASS_ORDER],
    color=[RISK_COLORS[c] for c in CLASS_ORDER],
    width=0.5, edgecolor='white', linewidth=1.2
)
for bar, cls in zip(bars, CLASS_ORDER):
    pct = counts[cls] / len(y_risk) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        f'{counts[cls]}  ({pct:.0f}%)',
        ha='center', fontsize=10, fontweight='bold'
    )
ax.set(title='Risk Class Distribution', ylabel='Count', xlabel='CiPA Class',
       ylim=(0, max(counts.values()) + 2))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Tabular IC50 summary ───────────────────────────────────────────────────────
print(f'\n{"Compound":<18} {"hERG (µM)":>10} {"Nav1.5 (µM)":>12} '
      f'{"Cav1.2 (µM)":>12}  Risk')
print('-' * 60)
for _, nm, h, nv, cl, risk in cipa_v:
    print(f'{nm:<18} {h:>10.3f} {nv:>12.3f} {cl:>12.3f}  {risk}')


---
## Step 3 — hERG IC50 Regression

**Why regression instead of binary classification?**  
Predicting a continuous IC50 gives the **safety margin** relative to the
therapeutic Cmax. A compound with IC50 = 2 µM and Cmax = 0.01 µM is safe;
the same IC50 with Cmax = 1 µM is dangerous. Binary 'blocker / not blocker'
loses this dose-response information.

**Why log₁₀ transform?**  
IC50 spans 4 orders of magnitude (0.001 – 127 µM). On a linear scale, a
residual of 5 µM is tiny for Sotalol (127 µM) but enormous for Astemizole
(0.001 µM). Log-transforming makes RMSE symmetric in relative (percentage) terms.

**Models compared:**

| Model | Strengths | Limitations |
|---|---|---|
| Random Forest | Robust to outliers, feature importance | Requires many trees for stability |
| SVR (RBF) | Effective in high-dim spaces | Sensitive to kernel/C/gamma tuning |
| XGBoost | Often best on tabular data, fast | Prone to overfit on n=27 |

**Cross-validation strategy:** 5-fold KFold (5–6 test compounds per fold).
With n=27 this is aggressive but acceptable — LOOCV would be more honest
but is unstable. Negative R² is expected: with only 27 samples,
2058-feature models will struggle. The exercise demonstrates the **workflow**.


In [ ]:
# =============================================================================
# Step 3 — hERG IC50 Regression: RF vs SVR vs XGBoost
# =============================================================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Three models with different inductive biases
regressors = [
    ('Random Forest',
     RandomForestRegressor(n_estimators=300, random_state=42)),
    ('SVR (RBF)',
     SVR(kernel='rbf', C=10, gamma='scale')),
    ('XGBoost',
     XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                  subsample=0.8, random_state=42, verbosity=0)),
]

print(f'{"Model":<20}  {"R² (CV)":>9}  {"RMSE (CV)":>10}  Interpretation')
print('-' * 75)

reg_results = {}
for model_name, reg in regressors:
    # cross_validate runs ONE fit per fold and returns multiple metrics —
    # more efficient than calling cross_val_score twice
    cv_out = cross_validate(
        reg, X_s, y_herg, cv=kf,
        scoring=['r2', 'neg_mean_squared_error'],
        return_train_score=False,
    )
    r2   = cv_out['test_r2'].mean()
    rmse = np.sqrt(-cv_out['test_neg_mean_squared_error'].mean())
    reg_results[model_name] = {'r2': r2, 'rmse': rmse}

    # R² < 0 means worse than predicting the mean — expected with n=27, 2058 features
    note = 'worse than mean predictor' if r2 < 0 else f'explains {r2*100:.0f}% variance'
    print(f'{model_name:<20}  {r2:>9.3f}  {rmse:>10.3f}     {note}')

print('\nNote: Negative R² on n=27 is expected. ')
print('In production (ChEMBL hERG: ~10 000 cpds), RF/XGBoost typically reach R²≈0.7.')

# ── Fit RF on all data — used ONLY for visualisation (parity + importances) ───
# IMPORTANT: this is a train-set fit. Points lying on y=x here does NOT mean
# good generalisation — it only confirms the model memorised the training data.
rf_full = RandomForestRegressor(n_estimators=300, random_state=42)
rf_full.fit(X_s, y_herg)
y_pred_train = rf_full.predict(X_s)

# ── Feature importances ───────────────────────────────────────────────────────
# RF uses Mean Decrease in Impurity (MDI) — how much each feature reduces
# the weighted variance across all tree splits.
# Physicochemical descriptors often dominate over individual ECFP4 bits
# because they encode global properties correlated with hERG binding.
importances = rf_full.feature_importances_

# argsort ascending; take last 20 (highest importance); keep ascending so
# barh() places the most important at the TOP (highest y position)
top20_idx = np.argsort(importances)[-20:]
bar_colors = ['#27ae60' if i >= 2048 else '#3498db' for i in top20_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: parity plot (train-set) ─────────────────────────────────────────────
ax = axes[0]
for nm, y_obs, y_hat, risk in zip(names, y_herg, y_pred_train, y_risk):
    ax.scatter(y_obs, y_hat, c=RISK_COLORS[risk], s=80, zorder=5, edgecolors='white', lw=0.5)
    ax.annotate(nm, (y_obs, y_hat), fontsize=6.5,
                xytext=(3, 3), textcoords='offset points', color='#333')

# Perfect prediction line (y = x)
lim = [y_herg.min() - 0.5, y_herg.max() + 0.5]
ax.plot(lim, lim, 'k--', lw=1, label='y = x  (perfect)')
for risk, colour in RISK_COLORS.items():
    ax.scatter([], [], c=colour, label=f'CiPA {risk}', s=60)
ax.set(xlim=lim, ylim=lim,
       xlabel='Observed  log₁₀(IC50 / µM)',
       ylabel='Predicted  log₁₀(IC50 / µM)',
       title='hERG IC50 — RF Train-Set Parity\n(memorisation check, not generalisation)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Right: top-20 feature importances ────────────────────────────────────────
ax = axes[1]
# top20_idx is ascending → barh gives most important at top (position 19)
ax.barh(range(20), importances[top20_idx], color=bar_colors)
ax.set_yticks(range(20))
ax.set_yticklabels([FEAT_NAMES[i] for i in top20_idx], fontsize=8)
ax.set(
    title='Top-20 Feature Importances (RF Regressor)\ngreen = physicochemical | blue = ECFP4 bit',
    xlabel='Mean Decrease in Impurity (MDI)'
)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('herg_regression.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Step 4 — CiPA Multi-Channel Risk Score

### The Hill Equation (Blockade Fraction)

For each channel we compute a **blockade fraction** `b` using the Hill equation:

$$b = \frac{1}{1 + \frac{IC_{50}}{C_{drug}}}$$

where `C_drug` is set to the CiPA consensus reference concentrations:
- hERG: 0.3 µM, Nav1.5: 1.0 µM, Cav1.2: 1.0 µM

At `C_drug = IC50`: b = 0.5 (50% blocked). At `C_drug << IC50`: b ≈ 0 (unblocked).

### Net CiPA Score

$$\text{score} = b_{hERG} - 0.5 \cdot b_{Nav} - 0.5 \cdot b_{CaL}$$

The negative terms reflect that Nav1.5 and Cav1.2 blockade *counteracts* the
QT-prolonging effect of hERG blockade. The 0.5 weights were derived empirically
from the CiPA reference dataset.

| Score | Risk class |
|-------|------------|
| > 0.5 | **High** |
| 0.2 – 0.5 | **Medium** |
| ≤ 0.2 | **Low** |

**Verapamil paradox explained:**  
b_hERG = 0.68, b_Nav = 0.17, b_CaL = 0.82 → score = 0.68 − 0.08 − 0.41 = **0.18 → Low**  
Despite being a strong hERG blocker, Cav1.2 offset makes it clinically safe.


In [ ]:
# =============================================================================
# Step 4 — CiPA Multi-Channel Blockade Score
# =============================================================================

# CiPA reference concentrations (µM) — from FDA/HESI consensus
C_HERG = 0.3   # at or near Cmax of typical CNS drugs
C_NAV  = 1.0
C_CAL  = 1.0


def blockade_fraction(ic50: float, c_ref: float) -> float:
    """Hill equation: fraction of channel blocked at concentration c_ref."""
    return 1.0 / (1.0 + ic50 / c_ref)


def cipa_risk_score(h_ic50: float, nav_ic50: float, cal_ic50: float) -> dict:
    """
    Compute CiPA net score and risk class from three IC50 values.

    Parameters
    ----------
    h_ic50   : hERG IC50 (µM)
    nav_ic50 : Nav1.5 IC50 (µM)
    cal_ic50 : Cav1.2 IC50 (µM)

    Returns
    -------
    dict with keys: b_hERG, b_Nav, b_CaL, net_score, risk_class
    """
    b_h = blockade_fraction(h_ic50,   C_HERG)
    b_n = blockade_fraction(nav_ic50, C_NAV)
    b_c = blockade_fraction(cal_ic50, C_CAL)

    # Protective channels offset the proarrhythmic hERG effect (weight 0.5 each)
    net = b_h - 0.5 * b_n - 0.5 * b_c

    risk = 'High' if net > 0.5 else ('Medium' if net > 0.2 else 'Low')
    return {'b_hERG': b_h, 'b_Nav': b_n, 'b_CaL': b_c,
            'net_score': net, 'risk_class': risk}


# ── Score every compound and tabulate results ──────────────────────────────────
print(f'{"Compound":<18} {"b_hERG":>7} {"b_Nav":>7} {"b_CaL":>7} '
      f'{"Score":>7} {"Pred":>8} {"True":>8}')
print('-' * 68)

score_rows = []
n_correct  = 0

for _, nm, h, nv, cl, true_risk in cipa_v:
    res  = cipa_risk_score(h, nv, cl)
    pred = res['risk_class']
    n_correct += (pred == true_risk)
    score_rows.append((nm, res['b_hERG'], res['b_Nav'], res['b_CaL'],
                       res['net_score'], pred, true_risk))
    tick = '✓' if pred == true_risk else '✗'
    print(f'{nm:<18} {res["b_hERG"]:>7.3f} {res["b_Nav"]:>7.3f} '
          f'{res["b_CaL"]:>7.3f} {res["net_score"]:>7.3f} '
          f'{pred:>8} {true_risk:>8}  {tick}')

print(f'\nCiPA rule accuracy: {n_correct}/{len(cipa_v)} = {n_correct/len(cipa_v)*100:.0f}%')
print('Mismatches are mainly Medium-class compounds — the most ambiguous cases.')

# ── Stacked bar: visual blockade balance per compound ─────────────────────────
# Positive bars = proarrhythmic hERG blockade
# Negative bars = protective Nav1.5 + Cav1.2 blockade (plotted below x-axis)

cmp_names, bhs, bns, bcs = zip(
    *[(r[0], r[1], r[2], r[3]) for r in score_rows]
)
x = np.arange(len(cmp_names))

fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(x, bhs,                         label='b_hERG (proarrhythmic)',
       color='#e74c3c', alpha=0.85)
ax.bar(x, [-0.5 * b for b in bns],     label='0.5×b_Nav (protective)',
       color='#3498db', alpha=0.75)
ax.bar(x, [-0.5 * b for b in bcs],
       bottom=[-0.5 * b for b in bns], label='0.5×b_CaL (protective)',
       color='#27ae60', alpha=0.75)

ax.axhline(0.5, color='k',    ls='--', lw=1.0, label='High threshold (0.5)')
ax.axhline(0.2, color='gray', ls=':',  lw=0.8, label='Medium threshold (0.2)')
ax.set_xticks(x)
ax.set_xticklabels(cmp_names, rotation=45, ha='right', fontsize=8)
ax.set(
    ylabel='Net blockade score',
    title='CiPA Multi-Channel Blockade Profile\n'
          '(Net = b_hERG − 0.5·b_Nav − 0.5·b_CaL)',
    ylim=(-0.65, 1.05)
)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cipa_blockade_profile.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Step 5 — ML Multi-Class Risk Classification

The CiPA rule uses experimentally measured IC50 values — useful for validation
but not for **virtual screening** where IC50 is unknown. Here we train a
Random Forest classifier to predict the risk class directly from SMILES
using the ECFP4 + physicochemical features from Step 1.

### Evaluation Strategy: Out-of-Fold (OOF) Predictions

With only 27 compounds and 3 classes, a standard train/test split would give
an unreliable estimate. Instead we use **3-fold stratified OOF**:

1. Split data into 3 folds, preserving class proportions in each (stratified)
2. For each fold: train on 2 folds, predict the held-out 1 fold
3. Collect all predictions — every compound was in the test set exactly once
4. Evaluate the full 27-compound prediction vector

**Balanced accuracy** is used (not plain accuracy) because the Medium class
has only 6 compounds. Plain accuracy would be ≥ 0.56 even by always predicting
High or Low — balanced accuracy normalises by class size.

**`class_weight='balanced'`** tells the RF to up-weight Medium-class samples
during tree construction, preventing the model from ignoring the minority class.


In [ ]:
# =============================================================================
# Step 5 — Multi-Class Risk Classification (RF Classifier)
# =============================================================================

CLASS_NAMES = ['High', 'Medium', 'Low']

clf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',   # up-weight minority class (Medium: only 6 compounds)
    random_state=42,
)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Collect out-of-fold predictions: each compound is predicted exactly once
y_oof = np.empty(len(y_cls), dtype=int)
for train_idx, test_idx in skf.split(X_s, y_cls):
    clf.fit(X_s[train_idx], y_cls[train_idx])
    y_oof[test_idx] = clf.predict(X_s[test_idx])

# ── Print metrics ─────────────────────────────────────────────────────────────
ba = balanced_accuracy_score(y_cls, y_oof)
print(f'Balanced Accuracy (3-fold OOF): {ba:.3f}')
print(f'  (random baseline = {1/3:.3f};  perfect = 1.000)\n')
print(classification_report(y_cls, y_oof, target_names=CLASS_NAMES))

# ── Figure: confusion matrix + feature importances ───────────────────────────
cm = confusion_matrix(y_cls, y_oof)

# Re-fit on all data for feature importance display only
# (OOF predictions above are the unbiased performance estimate)
clf_full = RandomForestClassifier(
    n_estimators=300, class_weight='balanced', random_state=42
).fit(X_s, y_cls)
importances_clf = clf_full.feature_importances_

# Top-15: ascending order so barh() shows most important at top
top15_idx  = np.argsort(importances_clf)[-15:]
bar_colors = ['#27ae60' if i >= 2048 else '#3498db' for i in top15_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: confusion matrix heat-map
ax = axes[0]
im = ax.imshow(cm, cmap='Blues', vmin=0)
ax.set(
    xticks=range(3), yticks=range(3),
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    xlabel='Predicted', ylabel='True',
    title=f'Confusion Matrix\nBalanced Accuracy = {ba:.2f}'
)
for i in range(3):
    for j in range(3):
        # Use white text on dark cells, black on light cells for readability
        txt_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center',
                fontsize=14, color=txt_color, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)

# Right: top-15 feature importances
ax = axes[1]
ax.barh(range(15), importances_clf[top15_idx], color=bar_colors)
ax.set_yticks(range(15))
ax.set_yticklabels([FEAT_NAMES[i] for i in top15_idx], fontsize=8)
ax.set(
    title='Top-15 Feature Importances (RF Classifier)\ngreen = physicochemical | blue = ECFP4 bit',
    xlabel='Mean Decrease in Impurity (MDI)'
)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('risk_classification.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Flag misclassified compounds ──────────────────────────────────────────────
# These are the highest-priority candidates for additional experimental data
# or structural alerts investigation
misclassified = [
    (names[i], INV_MAP[y_cls[i]], INV_MAP[y_oof[i]])
    for i in range(len(y_cls)) if y_cls[i] != y_oof[i]
]
if misclassified:
    print('\nMisclassified compounds (investigate structural novelty or OOD):'
          )
    print(f'{"Compound":<18} {"True":>10} {"Predicted":>12}')
    print('-' * 42)
    for nm, true_r, pred_r in misclassified:
        print(f'{nm:<18} {true_r:>10} {pred_r:>12}')


---
## Step 6 — Bootstrap Ensemble: Epistemic Uncertainty

### What is epistemic uncertainty?

**Epistemic** (model) uncertainty reflects the model's ignorance due to
limited or unrepresentative training data — it is reducible with more data.
This is distinct from **aleatoric** (data) uncertainty arising from
measurement noise, which cannot be reduced.

### Bootstrap ensemble method

We train **B = 200** independent RF regressors. Each is trained on a
different **bootstrap resample** (n=27 compounds drawn *with replacement*).
Each bootstrap sample includes ~63% of unique compounds on average
(some are repeated, some are left out).

$$\mu_i = \frac{1}{B}\sum_{b=1}^{B} \hat{f}_b(x_i)$$
$$\sigma_i = \text{std}_{b}[\hat{f}_b(x_i)]$$

- **High σ** → the B models disagree → compound is structurally dissimilar
  from most bootstrap training subsets → **out-of-domain (OOD)** → prioritise for wet-lab
- **Low σ** → the models agree → confident prediction

### Why not MC Dropout?

MC Dropout (Gal & Ghahramani 2016) approximates the same quantity via
Bayesian neural networks. It gives equivalent uncertainty estimates but
requires PyTorch and neural network training. For n=27, the bootstrap
ensemble is faster, more stable, and needs no extra dependencies.

### Error bar interpretation

Because IC50 is plotted on a **log scale**, ±σ in log space maps to
*asymmetric* intervals in linear (µM) space:
- Upper bar: `10^(μ+σ) − 10^μ`
- Lower bar: `10^μ − 10^(μ−σ)`


In [ ]:
# =============================================================================
# Step 6 — Bootstrap Ensemble Epistemic Uncertainty
# =============================================================================

rng = np.random.default_rng(seed=42)   # reproducible resampling
N   = len(y_herg)
B   = 200   # number of bootstrap replicates (200 gives stable std estimates)

boot_preds = np.zeros((B, N), dtype=np.float32)

for b in range(B):
    # Draw n indices WITH replacement — some compounds repeat, some are absent
    # On average, ~63% of unique compounds appear at least once per bootstrap
    idx_train = rng.integers(0, N, size=N)

    rf_b = RandomForestRegressor(
        n_estimators=50,   # 50 trees per replicate; fast and stable enough
        random_state=b,    # different seed per replicate for independent sampling
        n_jobs=1,
    )
    rf_b.fit(X_s[idx_train], y_herg[idx_train])

    # Predict on ALL compounds (including those not in this bootstrap)
    boot_preds[b] = rf_b.predict(X_s)

# ── Aggregate across replicates ───────────────────────────────────────────────
mu    = boot_preds.mean(axis=0)   # mean log10(IC50) prediction per compound
sigma = boot_preds.std(axis=0)    # std across 200 replicates = epistemic uncertainty

# Convert log10 → µM for display; asymmetric error bars because log→linear is nonlinear
mu_lin    = 10.0 ** mu
err_upper = 10.0 ** (mu + sigma) - mu_lin       # upper half of CI in µM
err_lower = np.clip(mu_lin - 10.0 ** (mu - sigma), 0, None)  # lower half (clip ≥ 0)

# ── Figure: predicted IC50 with uncertainty error bars ───────────────────────
fig, ax = plt.subplots(figsize=(15, 5))

ax.bar(
    range(N), mu_lin,
    yerr=[err_lower, err_upper],
    capsize=4,
    color=[RISK_COLORS[r] for r in y_risk],
    alpha=0.82,
    ecolor='#333',
    error_kw={'lw': 1.2},
)

# 1 µM is the conventional 'concern' threshold for hERG in vitro
ax.axhline(1.0, color='k', ls='--', lw=1.0, label='1 µM safety threshold')
for risk, colour in RISK_COLORS.items():
    ax.scatter([], [], c=colour, label=f'CiPA {risk}', s=60)

ax.set_xticks(range(N))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_yscale('log')   # log scale needed to see both 0.001 uM and 100 uM compounds
ax.set(
    ylabel='Predicted hERG IC50 (µM)  ±  bootstrap uncertainty',
    title=f'Bootstrap Ensemble Predictions  ({B} replicates × 50 trees)\n'
          'Error bars = ±1 std across replicates in log space',
)
ax.legend(fontsize=8, loc='upper left')
ax.grid(axis='y', alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('bootstrap_uncertainty_herg.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Uncertainty ranking table ─────────────────────────────────────────────────
# Sort descending by sigma — these compounds have the highest disagreement
# across bootstrap replicates → furthest from the training distribution
print('Uncertainty Ranking  (highest epistemic uncertainty → wetlab priority)')
print(f'{"Rank":<5} {"Compound":<18} {"µ log10(IC50)":>14} '
      f'{"σ":>7}  95% CI (µM)')
print('-' * 65)
ranked = sorted(zip(names, mu, sigma), key=lambda x: -x[2])
for rank, (nm, m, s) in enumerate(ranked, start=1):
    # Approximate 95% CI: µ ± 1.96σ in log space
    lo = 10 ** (m - 1.96 * s)
    hi = 10 ** (m + 1.96 * s)
    print(f'{rank:<5} {nm:<18} {m:>14.3f} {s:>7.3f}  [{lo:.3f}, {hi:.3f}]')


---
## Key Takeaways

### CiPA Framework (FDA/HESI/CSRC)

| Concept | Key insight |
|---------|-------------|
| hERG blockade alone | Insufficient predictor — Verapamil false positive |
| Nav1.5 / Cav1.2 offset | Protective channels shorten AP, counteracting hERG |
| Net CiPA score | Weighted sum: b_hERG − 0.5·b_Nav − 0.5·b_CaL |
| IC50 range in dataset | 0.001 – 127 µM = 5 log-decades → use log10 regression |

### Modelling Insights

| Topic | Recommendation |
|-------|---------------|
| Regression vs classification | Use regression for lead optimisation (gives safety margin) |
| Cross-validation | `cross_validate` for single-pass multi-metric CV |
| Class imbalance | `StratifiedKFold` + `class_weight='balanced'` |
| Feature importance | Always check whether physicochemical or ECFP4 features dominate |
| Uncertainty | Bootstrap std flags out-of-domain compounds for wet-lab follow-up |

### Regulatory Context

| Guideline | Scope |
|-----------|-------|
| **ICH E14** | Clinical QT/QTc study (required for NDA filing) |
| **ICH S7B** | Non-clinical cardiac electrophysiology (hERG patch-clamp mandatory) |
| **FDA CiPA** | Replaces single hERG endpoint with integrated 3-channel model |

### Industry Tools for hERG Prediction
- **hERGBoost** — gradient-boosted model, trained on ChEMBL (~10 000 cpds)
- **AttenhERG** — attention-based graph neural network
- **CardioTox** — web server: hERG + 5 additional cardiac endpoints
- **pkCSM / SwissADME / ADMETlab 3.0** — free multi-endpoint ADMET

### Next Steps
- **Notebook 04** — Neurotoxicity & BBB permeability (CNS safety)  
- **Notebook 05** — Hepatotoxicity & CYP450 inhibition  
- **Tutorial 10** — MEA functional assays for network neurotoxicity  
